# 09 - Preprocesado de tags semanticos

Este notebook prepara tags semanticos limpios de MovieLens para el recomendador hibrido final. El enfoque no conserva un tag solo porque no este en una blacklist: primero aplica rechazos duros explicables, despues clasifica cada tag en categorias semanticas utiles y finalmente aplica filtros estadisticos de fiabilidad. El resultado se guarda en `data/processed/` para que el notebook 08 pueda reutilizarlo sin recalcular la limpieza pesada.

## 1. Imports y rutas

In [12]:
import math
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('default')

ROOT = Path('..')
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
REPORTS_RESULTADOS = ROOT / 'reports' / 'resultados'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS_RESULTADOS.mkdir(parents=True, exist_ok=True)

TAGS_PATH = DATA_RAW / 'tags.csv'
TAGS_SEMANTIC_CLEAN_PATH = DATA_PROCESSED / 'tags_semantic_clean.csv'
TAG_SEMANTIC_STATS_PATH = DATA_PROCESSED / 'tag_semantic_stats.csv'
TAG_BLACKLIST_DETECTED_PATH = DATA_PROCESSED / 'tag_blacklist_detected.csv'
TAG_BLACKLIST_MANUAL_PATH = DATA_PROCESSED / 'tag_blacklist_manual.csv'

## 2. Carga de tags.csv

In [13]:
if not TAGS_PATH.exists():
    raise FileNotFoundError('No existe data/raw/tags.csv. Descarga MovieLens completo y colocalo en data/raw/.')

tags_raw = pd.read_csv(TAGS_PATH)

print('Shape:', tags_raw.shape)
print('Columnas:', list(tags_raw.columns))
print('Peliculas distintas:', tags_raw['movieId'].nunique() if 'movieId' in tags_raw.columns else 'movieId no existe')
print('Usuarios distintos:', tags_raw['userId'].nunique() if 'userId' in tags_raw.columns else 'userId no existe')
print('Tags unicos brutos:', tags_raw['tag'].nunique(dropna=True) if 'tag' in tags_raw.columns else 'tag no existe')
print('Porcentaje de tags nulos:', round(tags_raw['tag'].isna().mean() * 100, 4) if 'tag' in tags_raw.columns else 'tag no existe')

Shape: (2000072, 4)
Columnas: ['userId', 'movieId', 'tag', 'timestamp']
Peliculas distintas: 51323
Usuarios distintos: 15848
Tags unicos brutos: 140979
Porcentaje de tags nulos: 0.0008


## 3. Reglas duras manuales

In [14]:
HARD_REJECTION_GROUPS = {
    'ownership_watch_status': [
        'owned', 'own', 'seen', 'watched', 'watchlist', 'want to see', 'to watch',
    ],
    'admin_platform': [
        'dvd', 'bd-r', 'bdr', 'blu-ray', 'bluray', 'blue-ray', 'vhs', 'netflix',
        'amazon', 'hulu', 'criterion', 'criterion collection', 'library', 'collection',
        'on dvr', 'recorded', 'tivo',
    ],
    'ranking_or_canon': [
        'imdb', 'imdb top 250', 'top 250', 'top 100', 'afi', 'afi 100',
        '1001 movies', '1001 movies you must see before you die', 'national film registry',
        'library of congress', 'sight and sound', 'sight & sound', 'must see',
        'classic', 'classics', 'cult classic', 'cult film',
    ],
    'award': [
        'oscar', 'oscars', 'oscar winner', 'oscar nominated', 'academy award',
        'academy awards', 'award winner', 'award nominated', 'best picture',
        'golden globe', "palme d'or", 'golden palm', 'cannes', 'film festival',
        'festival winner',
    ],
    'generic_rating': [
        'good', 'great', 'best', 'bad', 'worst', 'boring', 'funny', 'very funny',
        'hilarious', 'overrated', 'underrated', 'favorite', 'favourite', 'favorites',
        'favourites', 'excellent', 'amazing', 'awesome', 'terrible', 'awful',
        'mediocre', 'masterpiece', 'brilliant', 'great acting', 'good acting',
        'bad acting', 'acting', 'well acted', 'well-acted', 'entertaining',
        'inspirational', 'touching', 'beautiful', 'beautifully filmed', 'well made',
        'well-made', 'slow paced',
    ],
    'metadata_weak': [
        'movie', 'movies', 'film', 'films', 'cinema', 'cinematic', 'based on',
        'based on book', 'based on a book', 'based on true story',
        'based on a true story', 'based on real events', 'true story', 'adapted from',
        'adaptation', 'remake', 'sequel', 'franchise', 'original', 'foreign',
        'foreign language', 'television', 'tv movie', 'made for tv', 'miniseries',
    ],
    'pure_genre': [
        'drama', 'comedy', 'action', 'thriller', 'romance', 'horror', 'sci-fi',
        'science fiction', 'crime', 'adventure', 'animation', 'children', 'fantasy',
        'mystery', 'documentary', 'war', 'western', 'musical', 'film-noir',
        'film noir', 'noir',
    ],
    'format_or_source': [
        'nudity', 'nudity (full frontal)', 'full frontal nudity', 'sex', 'sexual content',
    ],
    'person_or_studio_manual': [
        'christopher nolan', 'coen brothers', 'coen brother', 'tom hanks',
        'hans zimmer', 'jack nicholson', 'robert downey jr.', 'jordan peele',
        'morgan freeman', 'sandra bullock', 'jesse eisenberg', 'leonardo dicaprio',
        'brad pitt', 'quentin tarantino', 'martin scorsese', 'steven spielberg',
        'david fincher', 'ridley scott', 'pixar', 'disney', 'walt disney',
        'marvel', 'dc', 'dc comics', 'warner bros', 'dreamworks', 'studio ghibli',
        'ghibli', 'lucasfilm', 'hbo', 'netflix original',
    ],
}

manual_blacklist = (
    pd.DataFrame(
        {'tag_clean': tag, 'reason': reason}
        for reason, tags in HARD_REJECTION_GROUPS.items()
        for tag in tags
    )
    .drop_duplicates('tag_clean')
    .sort_values(['reason', 'tag_clean'])
)
manual_blacklist.to_csv(TAG_BLACKLIST_MANUAL_PATH, index=False)

HARD_REJECTION_REASON_BY_TAG = dict(zip(manual_blacklist['tag_clean'], manual_blacklist['reason']))

print('Reglas duras manuales guardadas en:', TAG_BLACKLIST_MANUAL_PATH)
print('Tags exactos en reglas duras manuales:', len(manual_blacklist))
display(manual_blacklist.head(30))

Reglas duras manuales guardadas en: ..\data\processed\tag_blacklist_manual.csv
Tags exactos en reglas duras manuales: 175


,tag_clean,reason
15,amazon,admin_platform
8,bd-r,admin_platform
9,bdr,admin_platform
10,blu-ray,admin_platform
12,blue-ray,admin_platform
11,bluray,admin_platform
20,collection,admin_platform
17,criterion,admin_platform
18,criterion collection,admin_platform
7,dvd,admin_platform


## 4. Normalizacion, rechazos duros y NLP ligero

In [15]:
DASH_TRANSLATION = str.maketrans({
    '?': '-', '?': '-', '?': '-', '?': '-', '?': '-', '?': '-', '?': '-',
})

CONTAINS_REJECTION_PATTERNS = [
    ('http', 'url_or_web'),
    ('www.', 'url_or_web'),
    ('imdb', 'ranking_or_canon'),
    ('oscar', 'award'),
    ('academy award', 'award'),
    ('criterion', 'admin_platform'),
    ('netflix', 'admin_platform'),
    ('dvd', 'admin_platform'),
    ('blu', 'admin_platform'),
    ('owned', 'ownership_watch_status'),
    ('watchlist', 'ownership_watch_status'),
    ('want to see', 'ownership_watch_status'),
    ('seen', 'ownership_watch_status'),
    ('watched', 'ownership_watch_status'),
    ('top 250', 'ranking_or_canon'),
    ('1001 movies', 'ranking_or_canon'),
    ('national film registry', 'ranking_or_canon'),
    ('library of congress', 'ranking_or_canon'),
    ('film registry', 'ranking_or_canon'),
    ('sight and sound', 'ranking_or_canon'),
    ('sight & sound', 'ranking_or_canon'),
    ('award nominated', 'award'),
    ('award winner', 'award'),
    ('based on true', 'metadata_weak'),
    ('based on a true', 'metadata_weak'),
    ('foreign language', 'metadata_weak'),
    ('made for tv', 'metadata_weak'),
    ('full frontal', 'format_or_source'),
]

SEMANTIC_CATEGORY_PATTERNS = {
    'narrative_structure': [
        'plot twist', 'twist ending', 'twist', 'surprise ending', 'ambiguous ending',
        'open ending', 'ending', 'nonlinear', 'non-linear', 'narrated', 'narration',
        'voiceover', 'multiple storylines', 'parallel storylines', 'unreliable narrator',
        'time loop', 'time travel', 'flashback', 'dream sequence', 'confusing',
        'cerebral', 'mindfuck', 'puzzle', 'mystery', 'slow burn', 'paced',
        'character study', 'dialogue-driven', 'dialogue driven',
    ],
    'theme': [
        'artificial intelligence', 'ai', 'robot', 'robots', 'android', 'cyborg',
        'dystopia', 'dystopian', 'post-apocalyptic', 'apocalypse', 'space',
        'astronaut', 'alien', 'aliens', 'time travel', 'time loop', 'memory',
        'identity', 'existentialism', 'existential', 'morality', 'moral', 'ethics',
        'grief', 'loss', 'loneliness', 'alienation', 'obsession', 'revenge',
        'redemption', 'guilt', 'trauma', 'mental illness', 'psychology',
        'psychological', 'dreams', 'afterlife', 'death', 'religion', 'faith',
        'philosophy', 'philosophical', 'social commentary', 'class', 'capitalism',
        'political', 'politics', 'war trauma', 'coming of age', 'friendship',
        'family relationships', 'father', 'mother', 'childhood', 'racism', 'gender',
        'feminism', 'media', 'technology', 'virtual reality', 'magical realism',
    ],
    'humor': [
        'dark comedy', 'black comedy', 'satire', 'satirical', 'absurd comedy',
        'absurd', 'deadpan', 'parody', 'spoof', 'quirky', 'offbeat', 'screwball',
        'slapstick', 'dry humor', 'surreal humor', 'witty', 'ironic', 'irony',
    ],
    'visual_style': [
        'stylized', 'style', 'visual', 'visually', 'cinematography',
        'beautifully shot', 'photography', 'colorful', 'minimalist', 'experimental',
        'expressionist', 'dreamlike', 'surreal', 'surrealism', 'animation style',
        'stop motion', 'rotoscoping', 'neo-noir', 'black and white', 'b&w',
        'long take', 'long takes', 'single take', 'shaky camera', 'handheld',
        'slow motion', 'practical effects',
    ],
    'tone_atmosphere': [
        'atmospheric', 'atmosphere', 'moody', 'dark', 'bleak', 'tense', 'tension',
        'melancholic', 'melancholy', 'dreamlike', 'haunting', 'creepy', 'eerie',
        'disturbing', 'bittersweet', 'sad', 'depressing', 'somber', 'gritty',
        'grim', 'weird', 'strange', 'hypnotic', 'meditative', 'quiet', 'lonely',
        'alienating', 'claustrophobic', 'paranoid', 'nostalgic', 'hope', 'hopeful',
        'heartwarming', 'feel-good', 'whimsical', 'absurd', 'absurdism', 'surreal',
        'surrealism',
    ],
    'emotional_cognitive_experience': [
        'thought-provoking', 'thought provoking', 'mind-bending', 'mind bending',
        'cerebral', 'intellectual', 'philosophical', 'emotional', 'moving',
        'powerful', 'intense', 'absorbing', 'engrossing', 'immersive',
        'challenging', 'complex', 'ambiguous', 'provocative', 'unsettling',
        'memorable',
    ],
    'intensity_darkness': [
        'violent', 'violence', 'brutal', 'disturbing', 'gory', 'bloody', 'dark',
        'bleak', 'scary', 'creepy', 'horror atmosphere', 'tense', 'suspenseful',
        'suspense', 'tragic', 'tragedy', 'depressing', 'nihilistic', 'revenge',
        'murder', 'serial killer',
    ],
}
SEMANTIC_CATEGORY_PRIORITY = [
    'narrative_structure',
    'theme',
    'humor',
    'visual_style',
    'tone_atmosphere',
    'emotional_cognitive_experience',
    'intensity_darkness',
]

PERSON_NAME_PROTECTED_EXPRESSIONS = {
    'time travel', 'dark comedy', 'black comedy', 'social commentary',
    'artificial intelligence', 'twist ending', 'coming of age', 'great soundtrack',
    'soundtrack', 'surreal', 'dreamlike', 'psychological', 'dystopia',
    'atmospheric', 'existentialism', 'neo-noir', 'noir', 'ambiguous ending',
    'morality', 'revenge', 'memory', 'identity', 'grief', 'lonely', 'alienation',
}
PERSON_NAME_SEMANTIC_WORDS = {
    'time', 'travel', 'dark', 'comedy', 'black', 'social', 'commentary',
    'artificial', 'intelligence', 'twist', 'ending', 'coming', 'age',
    'soundtrack', 'surreal', 'dream', 'dreamlike', 'psychological', 'dystopia',
    'atmospheric', 'existentialism', 'neo-noir', 'noir', 'ambiguous', 'morality',
    'revenge', 'memory', 'identity', 'grief', 'lonely', 'alienation',
}


def normalize_tag_text(tag):
    tag_clean = str(tag).lower().strip().translate(DASH_TRANSLATION)
    tag_clean = tag_clean.replace('_', ' ')
    tag_clean = re.sub(r'\s+', ' ', tag_clean).strip()
    tag_clean = tag_clean.strip(' \t\n\r\"\'`?????')
    tag_clean = re.sub(r'\s+', ' ', tag_clean).strip()
    if not tag_clean or tag_clean in {'nan', 'none', 'null'}:
        return None
    return tag_clean


def _contains_keyword(tag_clean, keyword):
    if not tag_clean or not keyword:
        return False
    keyword = keyword.lower()
    if re.fullmatch(r'[a-z0-9]+', keyword):
        return re.search(rf'(?<![a-z0-9]){re.escape(keyword)}(?![a-z0-9])', tag_clean) is not None
    return keyword in tag_clean


def classify_hard_rejection(tag_clean, tag_original=None):
    if tag_clean is None or pd.isna(tag_clean) or not str(tag_clean).strip():
        return 'null_or_empty'
    tag_clean = str(tag_clean).strip()

    if len(tag_clean) < 2:
        return 'too_short'
    if re.fullmatch(r'(19\d{2}|20[0-2]\d)', tag_clean):
        year = int(tag_clean)
        if 1900 <= year <= 2029:
            return 'exact_year'
    if re.fullmatch(r'(19[5-9]0s|20[0-2]0s|[5-9]0s)', tag_clean):
        return 'decade'
    if re.fullmatch(r'\d+(?:\.\d+)?', tag_clean):
        return 'numeric'
    if re.fullmatch(r'[\W_]+', tag_clean):
        return 'punctuation_only'

    exact_reason = HARD_REJECTION_REASON_BY_TAG.get(tag_clean)
    if exact_reason is not None:
        return exact_reason

    for pattern, reason in CONTAINS_REJECTION_PATTERNS:
        if pattern in tag_clean:
            return reason

    return None


def detect_semantic_category(tag_clean):
    if tag_clean is None or pd.isna(tag_clean):
        return None
    tag_clean = str(tag_clean).strip()
    for category in SEMANTIC_CATEGORY_PRIORITY:
        if any(_contains_keyword(tag_clean, keyword) for keyword in SEMANTIC_CATEGORY_PATTERNS[category]):
            return category
    return None


def detect_semantic_match_reason(tag_clean, semantic_category):
    if semantic_category is None:
        return None
    for keyword in SEMANTIC_CATEGORY_PATTERNS[semantic_category]:
        if _contains_keyword(tag_clean, keyword):
            return keyword
    return None


def looks_like_person_name(tag_original, tag_clean):
    if tag_clean is None or tag_clean in PERSON_NAME_PROTECTED_EXPRESSIONS:
        return False
    if any(word in tag_clean.split() for word in PERSON_NAME_SEMANTIC_WORDS):
        return False

    original = str(tag_original).strip().translate(DASH_TRANSLATION)
    tokens = [token.strip('.,:;!?()[]{}\"\'`?????') for token in original.split()]
    tokens = [token for token in tokens if token]
    if len(tokens) not in {2, 3}:
        return False

    capitalized_tokens = sum(bool(re.match(r'^[A-Z??????][A-Za-z????????????\'-]+$', token)) for token in tokens)
    return capitalized_tokens >= 2

## 5. Crear tags normalizados y categorias semanticas

In [16]:
tags_work = tags_raw.copy()
tags_work['tag_original'] = tags_work['tag'].astype(str)
tags_work['tag_clean'] = tags_work['tag'].apply(normalize_tag_text)
tags_work['hard_rejection_reason'] = tags_work.apply(
    lambda row: classify_hard_rejection(row['tag_clean'], row['tag_original']), axis=1
)
tags_work['semantic_category'] = tags_work['tag_clean'].apply(detect_semantic_category)
tags_work['semantic_match_reason'] = tags_work.apply(
    lambda row: detect_semantic_match_reason(row['tag_clean'], row['semantic_category']), axis=1
)

tags_work['person_name_like'] = tags_work.apply(
    lambda row: (
        pd.isna(row['hard_rejection_reason'])
        and pd.isna(row['semantic_category'])
        and looks_like_person_name(row['tag_original'], row['tag_clean'])
    ),
    axis=1,
)
tags_work['semantic_rejection_reason'] = np.select(
    [tags_work['person_name_like'], tags_work['semantic_category'].isna()],
    ['person_name_like', 'semantic_category_missing'],
    default=None,
)
tags_work.loc[tags_work['hard_rejection_reason'].notna(), 'semantic_rejection_reason'] = None

tags_work['preliminary_keep'] = tags_work['hard_rejection_reason'].isna() & tags_work['semantic_category'].notna()
tags_work_nonnull = tags_work[tags_work['tag_clean'].notna()].copy()

dedup_cols = ['movieId', 'tag_clean']
if 'userId' in tags_work_nonnull.columns:
    dedup_cols = ['userId', 'movieId', 'tag_clean']

tags_work_dedup = tags_work_nonnull.drop_duplicates(dedup_cols).copy()
preliminary_tags = tags_work_dedup[tags_work_dedup['preliminary_keep']].copy()

print('Filas originales:', len(tags_raw))
print('Tags unicos brutos:', tags_raw['tag'].nunique(dropna=True))
print('Filas con tag_clean valido:', len(tags_work_nonnull))
print('Filas deduplicadas con tag_clean valido:', len(tags_work_dedup))
print('Tags preliminares con semantic_category util:', preliminary_tags['tag_clean'].nunique())
print('Filas preliminares:', len(preliminary_tags))

KeyError: nan

## 6. Estadisticas globales y fiabilidad

In [ ]:
def mode_or_first(values):
    values = values.dropna()
    if values.empty:
        return None
    mode = values.mode()
    return mode.iloc[0] if not mode.empty else values.iloc[0]


def build_tag_usage_stats(df, n_movies_total_with_tags):
    stats = df.groupby('tag_clean', as_index=False).agg(
        n_uses=('tag_clean', 'size'),
        n_movies=('movieId', 'nunique'),
    )
    if 'userId' in df.columns:
        user_stats = df.groupby('tag_clean', as_index=False).agg(n_users=('userId', 'nunique'))
        top_user_stats = (
            df.groupby(['tag_clean', 'userId'])
            .size()
            .groupby('tag_clean')
            .max()
            .rename('top_user_uses')
            .reset_index()
        )
        stats = stats.merge(user_stats, on='tag_clean', how='left').merge(top_user_stats, on='tag_clean', how='left')
    else:
        stats['n_users'] = np.nan
        stats['top_user_uses'] = np.nan

    stats['top_user_share'] = stats['top_user_uses'] / stats['n_uses']
    stats['pct_movies'] = stats['n_movies'] / max(n_movies_total_with_tags, 1)
    stats['idf'] = np.log((1 + n_movies_total_with_tags) / (1 + stats['n_movies'])) + 1
    return stats


def first_statistical_rejection(row):
    if not row['preliminary_keep'] or row['passes_statistical_filters']:
        return None
    if row['fail_low_n_movies']:
        return 'low_n_movies'
    if row['fail_low_n_users']:
        return 'low_n_users'
    if row['fail_high_pct_movies']:
        return 'high_pct_movies'
    if row['fail_high_top_user_share']:
        return 'high_top_user_share'
    return None

n_movies_total_with_tags = preliminary_tags['movieId'].nunique()
base_stats = build_tag_usage_stats(tags_work_dedup, n_movies_total_with_tags)

tag_attributes = tags_work_dedup.groupby('tag_clean').agg(
    semantic_category=('semantic_category', mode_or_first),
    semantic_match_reason=('semantic_match_reason', mode_or_first),
    preliminary_keep=('preliminary_keep', 'max'),
    hard_rejection_reason=('hard_rejection_reason', mode_or_first),
    semantic_rejection_reason=('semantic_rejection_reason', mode_or_first),
).reset_index()

tag_stats = base_stats.merge(tag_attributes, on='tag_clean', how='left')
tag_stats['preliminary_keep'] = tag_stats['preliminary_keep'].fillna(False).astype(bool)
tag_stats['fail_low_n_movies'] = tag_stats['n_movies'] < 5
tag_stats['fail_low_n_users'] = tag_stats['n_users'] < 5 if 'userId' in tags_work_dedup.columns else False
tag_stats['fail_high_pct_movies'] = tag_stats['pct_movies'] > 0.05
tag_stats['fail_high_top_user_share'] = tag_stats['top_user_share'] > 0.60 if 'userId' in tags_work_dedup.columns else False

tag_stats['passes_statistical_filters'] = (
    tag_stats['preliminary_keep']
    & ~tag_stats['fail_low_n_movies']
    & ~tag_stats['fail_low_n_users']
    & ~tag_stats['fail_high_pct_movies']
    & ~tag_stats['fail_high_top_user_share']
)
tag_stats['is_reliable_tag'] = tag_stats['preliminary_keep'] & tag_stats['passes_statistical_filters']
tag_stats['rejection_reason_statistical'] = tag_stats.apply(first_statistical_rejection, axis=1)

reliable_tag_stats = tag_stats[tag_stats['is_reliable_tag']].copy()

print('Tags aceptados antes de fiabilidad:', int(tag_stats['preliminary_keep'].sum()))
print('Tags finales fiables:', len(reliable_tag_stats))
print('Eliminados por low_n_movies:', int((tag_stats['preliminary_keep'] & tag_stats['fail_low_n_movies']).sum()))
print('Eliminados por low_n_users:', int((tag_stats['preliminary_keep'] & tag_stats['fail_low_n_users']).sum()))
print('Eliminados por high_pct_movies:', int((tag_stats['preliminary_keep'] & tag_stats['fail_high_pct_movies']).sum()))
print('Eliminados por high_top_user_share:', int((tag_stats['preliminary_keep'] & tag_stats['fail_high_top_user_share']).sum()))

display(reliable_tag_stats.sort_values('n_uses', ascending=False).head(30))
display(reliable_tag_stats.sort_values('n_movies', ascending=False).head(30))

## 7. Crear tag_semantic_stats.csv

In [ ]:
TAG_STATS_COLUMNS = [
    'tag_clean',
    'semantic_category',
    'semantic_match_reason',
    'n_uses',
    'n_movies',
    'n_users',
    'top_user_uses',
    'top_user_share',
    'pct_movies',
    'idf',
    'passes_statistical_filters',
    'is_reliable_tag',
    'hard_rejection_reason',
    'semantic_rejection_reason',
    'rejection_reason_statistical',
]

tag_stats_export = tag_stats[TAG_STATS_COLUMNS].sort_values(
    ['is_reliable_tag', 'semantic_category', 'n_movies', 'n_uses'],
    ascending=[False, True, False, False],
)
tag_stats_export.to_csv(TAG_SEMANTIC_STATS_PATH, index=False)

print('Estadisticas guardadas en:', TAG_SEMANTIC_STATS_PATH)
print('Filas:', len(tag_stats_export))

## 8. Crear tags_semantic_clean.csv

In [ ]:
reliable_tag_details = tag_stats.loc[
    tag_stats['is_reliable_tag'],
    ['tag_clean', 'semantic_category', 'idf', 'n_uses', 'n_movies', 'n_users', 'top_user_share'],
].copy()

tags_semantic_clean = (
    preliminary_tags[['movieId', 'tag_clean']]
    .drop_duplicates(['movieId', 'tag_clean'])
    .merge(reliable_tag_details, on='tag_clean', how='inner')
    .sort_values(['movieId', 'tag_clean'])
)

tags_semantic_clean.to_csv(TAGS_SEMANTIC_CLEAN_PATH, index=False)

print('Tags semanticos limpios guardados en:', TAGS_SEMANTIC_CLEAN_PATH)
print('Filas finales tags_semantic_clean:', len(tags_semantic_clean))
print('Tags unicos finales:', tags_semantic_clean['tag_clean'].nunique())
print('Peliculas con tags semanticos limpios:', tags_semantic_clean['movieId'].nunique())

## 9. Crear tag_blacklist_detected.csv

In [ ]:
BLACKLIST_STATS_COLUMNS = [
    'tag_clean', 'semantic_category', 'n_uses', 'n_movies', 'n_users',
    'top_user_share', 'pct_movies', 'idf',
]

hard_detected = tag_stats[tag_stats['hard_rejection_reason'].notna()][BLACKLIST_STATS_COLUMNS + ['hard_rejection_reason']].copy()
hard_detected['rejection_source'] = 'hard_rules'
hard_detected = hard_detected.rename(columns={'hard_rejection_reason': 'rejection_reason'})

semantic_detected = tag_stats[
    tag_stats['hard_rejection_reason'].isna()
    & tag_stats['semantic_rejection_reason'].notna()
][BLACKLIST_STATS_COLUMNS + ['semantic_rejection_reason']].copy()
semantic_detected['rejection_source'] = 'semantic_rules'
semantic_detected = semantic_detected.rename(columns={'semantic_rejection_reason': 'rejection_reason'})

statistical_detected = tag_stats[
    tag_stats['rejection_reason_statistical'].notna()
][BLACKLIST_STATS_COLUMNS + ['rejection_reason_statistical']].copy()
statistical_detected['rejection_source'] = 'statistical_rules'
statistical_detected = statistical_detected.rename(columns={'rejection_reason_statistical': 'rejection_reason'})

tag_blacklist_detected = pd.concat([hard_detected, semantic_detected, statistical_detected], ignore_index=True)
tag_blacklist_detected = tag_blacklist_detected[
    ['tag_clean', 'rejection_source', 'rejection_reason', 'semantic_category', 'n_uses',
     'n_movies', 'n_users', 'top_user_share', 'pct_movies', 'idf']
].sort_values(['rejection_source', 'rejection_reason', 'n_movies', 'n_uses'], ascending=[True, True, False, False])

tag_blacklist_detected.to_csv(TAG_BLACKLIST_DETECTED_PATH, index=False)

print('Blacklist detectada guardada en:', TAG_BLACKLIST_DETECTED_PATH)
print('Filas:', len(tag_blacklist_detected))
display(tag_blacklist_detected.head(30))

## 10. Diagnostico final

In [ ]:
def safe_sample(df, n=50, random_state=42):
    if df.empty:
        return df
    return df.sample(min(n, len(df)), random_state=random_state)

print('Resumen')
print('Filas originales:', len(tags_raw))
print('Tags unicos brutos:', tags_raw['tag'].nunique(dropna=True))
print('Tags con tag_clean valido:', tags_work_nonnull['tag_clean'].nunique())
print('Tags preliminares con semantic_category util:', preliminary_tags['tag_clean'].nunique())
print('Tags finales fiables:', reliable_tag_stats['tag_clean'].nunique())
print('Filas finales en tags_semantic_clean:', len(tags_semantic_clean))
print('Peliculas con tags semanticos limpios:', tags_semantic_clean['movieId'].nunique())

print('\nDistribucion de semantic_category por tags unicos finales')
display(reliable_tag_stats['semantic_category'].value_counts(dropna=False).rename_axis('semantic_category').reset_index(name='n_tags'))

print('\nDistribucion de semantic_category por filas finales')
display(tags_semantic_clean['semantic_category'].value_counts(dropna=False).rename_axis('semantic_category').reset_index(name='n_rows'))

print('\nRechazos duros')
display(tags_work_dedup['hard_rejection_reason'].value_counts(dropna=False).rename_axis('hard_rejection_reason').reset_index(name='n_rows'))

semantic_missing_tags = tag_stats[tag_stats['semantic_rejection_reason'] == 'semantic_category_missing']
person_like_tags = tag_stats[tag_stats['semantic_rejection_reason'] == 'person_name_like']
print('Tags con semantic_category_missing:', len(semantic_missing_tags))
print('Tags rechazados por person_name_like:', len(person_like_tags))
print('Tags rechazados por low_n_movies:', int((tag_stats['preliminary_keep'] & tag_stats['fail_low_n_movies']).sum()))
print('Tags rechazados por low_n_users:', int((tag_stats['preliminary_keep'] & tag_stats['fail_low_n_users']).sum()))
print('Tags rechazados por high_pct_movies:', int((tag_stats['preliminary_keep'] & tag_stats['fail_high_pct_movies']).sum()))
print('Tags rechazados por high_top_user_share:', int((tag_stats['preliminary_keep'] & tag_stats['fail_high_top_user_share']).sum()))

for category in SEMANTIC_CATEGORY_PRIORITY:
    category_tags = reliable_tag_stats[reliable_tag_stats['semantic_category'] == category]
    if category_tags.empty:
        continue
    print(f'\nTop 20 tags finales por categoria: {category}')
    display(category_tags.sort_values(['n_movies', 'n_uses'], ascending=False).head(20))

print('\nMuestra aleatoria de 50 tags finales aceptados')
display(safe_sample(reliable_tag_stats[['tag_clean', 'semantic_category', 'n_movies', 'n_uses']], 50).sort_values('tag_clean'))

print('\nMuestra aleatoria de 50 tags rechazados por semantic_category_missing')
display(safe_sample(semantic_missing_tags[['tag_clean', 'n_movies', 'n_uses']], 50).sort_values('tag_clean'))

print('\nMuestra aleatoria de 50 tags rechazados por person_name_like')
display(safe_sample(person_like_tags[['tag_clean', 'n_movies', 'n_uses']], 50).sort_values('tag_clean'))

problematic_tags = [
    'national film registry', 'pixar', 'disney', 'brilliant', 'foreign',
    'based on a true story', '1980s', '1990s', 'nudity (full frontal)',
    'dave franco', 'isla fisher', 'christopher nolan', 'coen brothers',
    'tom hanks', 'hans zimmer', 'great acting', 'cult film', 'owned',
    'imdb top 250', 'oscar', 'criterion', 'dvd', 'netflix',
]
final_tag_set = set(tags_semantic_clean['tag_clean'])
problematic_check = pd.DataFrame({
    'tag_clean': problematic_tags,
    'present_in_tags_semantic_clean': [tag in final_tag_set for tag in problematic_tags],
})
print('\nComprobacion de tags problematicos')
display(problematic_check)

useful_tags = [
    'surrealism', 'dreamlike', 'existentialism', 'psychological',
    'ambiguous ending', 'neo-noir', 'atmospheric', 'time travel',
    'thought-provoking', 'dark comedy', 'black comedy', 'social commentary',
    'plot twist', 'twist ending', 'mindfuck', 'bittersweet', 'quirky',
    'satirical', 'cerebral', 'magical realism', 'tense', 'morality',
]
raw_tag_set = set(tags_work_nonnull['tag_clean'])
useful_check = pd.DataFrame({
    'tag_clean': useful_tags,
    'exists_in_tags_csv': [tag in raw_tag_set for tag in useful_tags],
    'present_in_tags_semantic_clean': [tag in final_tag_set for tag in useful_tags],
})
print('\nComprobacion de tags utiles')
display(useful_check)

existing_useful = useful_check[useful_check['exists_in_tags_csv']]
if len(existing_useful) and existing_useful['present_in_tags_semantic_clean'].mean() < 0.35:
    warnings.warn('Muchos tags utiles existentes quedaron fuera; revisa patrones o filtros estadisticos.')

## 11. Criterios de aceptacion

Al ejecutarse completo, este notebook debe generar:

- `data/processed/tags_semantic_clean.csv`
- `data/processed/tag_semantic_stats.csv`
- `data/processed/tag_blacklist_detected.csv`
- `data/processed/tag_blacklist_manual.csv`

`tags_semantic_clean.csv` conserva las columnas base que usa el notebook 08 (`movieId`, `tag_clean`, `idf`) y anade `semantic_category` para auditoria. No usa APIs externas, LightFM, spaCy, sentence-transformers ni dependencias nuevas.